# 03c — Joint time–wavelength Gaussian-process features

Fit a joint Matern-3/2 GP to all available passbands. Fix the wavelength scale
at 600 nm and constrain the temporal scale to 5–70 days. Normalize flux and
flux errors by the object's maximum absolute flux, then restore the original
units for prediction. Require eight observations spanning at least three bands.

Use a common peak phase for cross-band comparisons. The 53 descriptors include
colors, shape, peak offsets and secondary structure. Requested phases outside
the observed interval receive a zero sentinel rather than extrapolated measurements.


## 0. Configuration


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, tempfile
from pathlib import Path


os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'mpl_cache'))

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'config.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'config.py').exists():
    raise FileNotFoundError('No  encontró config.py desde the directory of ejecución')
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
from src.features import gp2d
from src.features.cache import write_cache, load_cache


## 1. Load the cleaned curves


In [ ]:
df = pd.read_csv(config.TRAIN_CLEAN_CSV)
metadata_table = pd.read_csv(config.METADATA_CLEAN_CSV)[['object_id', 'target']]
print(df.shape, '·', df.object_id.nunique(), 'objetos')
df.head()


## 2. Inspect a joint GP fit


In [ ]:

oid = 252646

light_curve = df[df.object_id == oid]
m = gp2d.fit_gp2d(light_curve)
feats = gp2d.object_gp2d_features(light_curve)
t0, tt = m['t0'], m['tt']


tgrid = np.linspace(0, tt.max(), 160)
waves = np.array([gp2d.WAVELENGTHS[i] for i in range(6)])
wgrid = np.linspace(waves.min(), waves.max(), 120)
T, W = np.meshgrid(tgrid, wgrid)
mu = m['gp'].predict(m['fn'], np.column_stack([T.ravel(), W.ravel()]),
                     return_cov=False, return_var=False).reshape(T.shape) * m['scale']

cols = {'u': 'violet', 'g': 'green', 'r': 'red', 'i': 'darkred', 'z': 'saddlebrown', 'y': 'black'}
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
im = ax[0].pcolormesh(tgrid, wgrid, mu, cmap='viridis', shading='auto')
for i in range(6):
    s = light_curve[light_curve.passband == i]
    ax[0].scatter(s.mjd - t0, [gp2d.WAVELENGTHS[i]] * len(s), s=14, color='white', edgecolor='k', lw=.3, alpha=.7)
    ax[0].text(tt.max() * 1.01, gp2d.WAVELENGTHS[i], 'ugrizy'[i], va='center', fontsize=9)
ax[0].axvline(feats['gp2d_t_peak'], color='w', ls='--', lw=1)
ax[0].set(xlabel='Days since first observation', ylabel='Wavelength (Å)',
          title='GP2D surface: flux(time, $\\lambda$)')
fig.colorbar(im, ax=ax[0], label='Flux')

tline = np.linspace(0, tt.max(), 200)
for i, b in enumerate('ugrizy'):
    ax[1].plot(tline, gp2d._predict(m, tline + t0, b), color=cols[b], lw=1.6, label=b)
    s = light_curve[light_curve.passband == i]
    ax[1].errorbar(s.mjd - t0, s.flux, yerr=s.flux_err, fmt='o', ms=3, color=cols[b], alpha=.45, lw=.6)
ax[1].axvline(feats['gp2d_t_peak'], color='gray', ls='--', lw=1, label='Peak')
ax[1].set(xlabel='Days since first observation', ylabel='Flux',
          title='Light curves by band (GP2D mean)')
ax[1].legend(ncol=3, fontsize=8)
fig.suptitle(f"GP2D fit — object #{oid}  ·  colours at peak: g-r={feats['gp2d_color_gr']:.2f}, "
             f"r-i={feats['gp2d_color_ri']:.2f}, i-z={feats['gp2d_color_iz']:.2f}", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(config.FIGURE_DIR / 'gp2d_superficie_ejemplo.png', dpi=150, bbox_inches='tight')
fig.savefig(config.FIGURE_DIR / 'gp2d_superficie_ejemplo.pdf', bbox_inches='tight')
plt.show()


## 3. Extract features for one object


In [ ]:
pd.Series(gp2d.object_gp2d_features(light_curve)).to_frame('valor')


## 4. Build the GP2D feature table


In [ ]:





catalogo_desde_cache = config.FEATURES_GP2D_CSV.exists()
if catalogo_desde_cache:
    tabla_gp2d = load_cache(config.FEATURES_GP2D_CSV, 'gp2d')
    print('Cache loaded ->', config.FEATURES_GP2D_CSV)
else:
    tabla_gp2d = gp2d.build_gp2d_table(df)
print('tabla_gp2d:', tabla_gp2d.shape)
tabla_gp2d.head()


## 5. Check the schema and numerical values


In [ ]:
assert tabla_gp2d.object_id.is_unique, 'object_id duplicado'
assert tabla_gp2d.isna().sum().sum() == 0, 'The table contains NaN values'
n_sin = (tabla_gp2d[gp2d.GP2D_FEATURES].abs().sum(axis=1) == 0).sum()
print(f"{tabla_gp2d.shape[0]:,} objects · {tabla_gp2d.shape[1]-1} features GP 2D")
print(f"Objects without a fit (all zeros; insufficient observations/bands): {n_sin}")


## 6. Save the feature cache


In [ ]:
if catalogo_desde_cache:
    print('Retained catalogue ->', config.FEATURES_GP2D_CSV)
else:
    write_cache(tabla_gp2d, config.FEATURES_GP2D_CSV, 'gp2d', producer_name='notebooks/03c_gp2d_features.ipynb')
    print('saved ->', config.FEATURES_GP2D_CSV)


## 7. Classification input

Combine this block with statistics, GP1D descriptors and 24 metadata-conditioned
transformations. The complete E2-GP representation contains 304 numerical inputs.
